# Labeling the dataset by a transformer model

Loads the fine-tuned transformer model and labels the **entire** dataset using the
efficient batched method. The model is produced by `transformers_model_finetuner.py`.

## Setup

In [1]:
import pandas as pd
import os

# gfx1150 (AMD Radeon 840M) has no ROCm kernels; present it as gfx1100. Set before torch import; no-op off ROCm.
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = '../data'
MODELS_DIR = '../models'

model_dir = f'{MODELS_DIR}/attempt2-20-classes/hf_transformer_model'
TRAIN_CSV = f'{DATA_DIR}/labeled_balanced_20.csv'  # the CSV the model was trained on
INPUT_CSV = f'{DATA_DIR}/labeled.csv'
OUTPUT_CSV = f'{DATA_DIR}/filtered_events_class_with_predicted.csv'

## Loading the model

In [3]:
print(f"\n--- Loading the model from {model_dir} ---")
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

if torch.backends.mps.is_available():
    load_device = torch.device("mps")
elif torch.cuda.is_available():
    load_device = torch.device("cuda")
else:
    load_device = torch.device("cpu")

loaded_model.to(load_device)
loaded_model.eval()

print(f"Using device: {load_device}")
print("Model and tokenizer loaded successfully!")


--- Loading the model from ../models/attempt2-20-classes/hf_transformer_model ---


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21865.73it/s]


Using device: cuda
Model and tokenizer loaded successfully!


## Loading the dataset

In [4]:
dataset = pd.read_csv(INPUT_CSV)
len(dataset)

/tmp/ipykernel_218163/4017719280.py:1: DtypeWarning: Columns (0: civilian_targeting) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(INPUT_CSV)


214232

In [5]:
# Read the class names straight from the model config (set at train time by
# transformers_model_finetuner.py). Fall back to the training CSV only for older
# models saved with generic LABEL_0..N names.
id2label = {int(k): v for k, v in loaded_model.config.id2label.items()}

if all(str(v).startswith("LABEL_") for v in id2label.values()):
    _train_df = pd.read_csv(TRAIN_CSV)
    _train_df = _train_df[~_train_df['class'].isin(['NoN', 'unknown'])]
    id2label = {i: name for i, name in enumerate(sorted(_train_df['class'].unique()))}
    print(f"Model has generic labels; reconstructed {len(id2label)} from {TRAIN_CSV}")
else:
    print(f"Using {len(id2label)} labels from the model config")

assert len(id2label) == loaded_model.config.num_labels
for i in sorted(id2label):
    print(f"  {i}: {id2label[i]}")

Using 20 labels from the model config
  0: animal welfare
  1: blm
  2: climate
  3: culture
  4: discrimination
  5: education
  6: environment
  7: farmers
  8: health care
  9: housing
  10: immigration
  11: labor rights
  12: lgbtq
  13: palestine-israel conflict
  14: pandemic
  15: policies & politics
  16: public services
  17: ukraine-russia war
  18: unjust law enforcement
  19: women rights


## Batched labeling over the whole dataset (fast)

Batched inference with length-sorted dynamic padding, bf16 autocast on CUDA, and
`inference_mode`. Original row order is restored before writing.

In [ ]:
MAX_LEN_HF = 128
BATCH_SIZE = 256  # lower this if you hit out-of-memory

texts = [str(t) for t in dataset['clean_notes'].tolist()]
n = len(texts)

order = sorted(range(n), key=lambda i: len(texts[i]))  # group similar lengths for tight padding
predictions = [None] * n

use_bf16 = (load_device.type == 'cuda')

for start in tqdm(range(0, n, BATCH_SIZE), desc="Labeling dataset (batched)"):
    batch_pos = order[start:start + BATCH_SIZE]
    batch_texts = [texts[i] for i in batch_pos]

    inputs = loaded_tokenizer(
        batch_texts,
        return_tensors='pt',
        padding='longest',
        truncation=True,
        max_length=MAX_LEN_HF,
    )
    inputs = {key: val.to(load_device) for key, val in inputs.items()}

    with torch.inference_mode():
        with torch.autocast(device_type=load_device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = loaded_model(**inputs).logits
        predicted = torch.argmax(logits, dim=1).cpu().tolist()

    for pos, cls_idx in zip(batch_pos, predicted):
        predictions[pos] = id2label[cls_idx]

dataset['predicted_class'] = predictions
dataset.to_csv(OUTPUT_CSV, index=False)
print(f"Saved predictions to {OUTPUT_CSV}")